# Exploratory Analysis of Global Earthquake-Tsunami Events (2001–2022)

## Objective
To conduct an in-depth exploratory data analysis (EDA) using Python on the Global Earthquake-Tsunami Risk Assessment Dataset. The goal is to identify patterns, trends, and differences between tsunami-generating and non-tsunami earthquakes based on seismic features.

This analysis uses statistical summaries and visualizations implemented with Matplotlib and Seaborn to explore:
- **Time-Based Patterns**: How earthquakes and tsunamis have evolved over 22 years
- **Seismic Characteristics**: Magnitude and depth distributions
- **Geographic Distribution**: Spatial clustering and regional patterns
- **Comparative Analysis**: Key differences between tsunami vs. non-tsunami events
- **Correlation Insights**: Relationships between seismic variables

---

## Section 1: Import Required Libraries

In this section, we import all necessary libraries for data manipulation, numerical operations, and visualization.

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# Configure visualization settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Set figure size and font sizes for better readability
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9

print("✓ All libraries imported successfully!")
print(f"  - Pandas version: {pd.__version__}")
print(f"  - NumPy version: {np.__version__}")
print(f"  - Matplotlib version: {plt.matplotlib.__version__}")
print(f"  - Seaborn version: {sns.__version__}")

## Section 2: Load and Inspect the Dataset

Load the Global Earthquake-Tsunami Risk Assessment Dataset and explore its structure, data types, and basic statistics.

In [ ]:
# Load the dataset
df = pd.read_csv('../data/earthquakes_dataset.csv')

# Convert Date column to datetime
df['Date'] = pd.to_datetime(df['Date'])

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"\n📊 Dataset Shape: {df.shape}")
print(f"   - Total Records: {df.shape[0]}")
print(f"   - Total Features: {df.shape[1]}\n")

print("📋 Column Information:")
print(df.dtypes)

print("\n" + "=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
missing = df.isnull().sum()
print(f"\nMissing Values:\n{missing[missing > 0] if missing.sum() > 0 else 'No missing values! ✓'}")

print("\n" + "=" * 80)
print("BASIC STATISTICS")
print("=" * 80)
print("\n📈 Numerical Columns Summary:")
print(df.describe().round(3))

print("\n" + "=" * 80)
print("CATEGORICAL DATA SUMMARY")
print("=" * 80)
print(f"\n🌊 Tsunami Distribution:\n{df['Tsunami'].value_counts()}")
print(f"\n   Tsunami Percentage: {(df['Tsunami'] == 'Yes').sum() / len(df) * 100:.2f}%")

print("\n" + "=" * 80)
print("FIRST FEW RECORDS")
print("=" * 80)
print(df.head(10))

## Section 3: Data Cleaning and Preprocessing

Handle missing values, remove duplicates, and create derived features for analysis.

In [ ]:
# Create a working copy
df_clean = df.copy()

print("=" * 80)
print("DATA CLEANING AND PREPROCESSING")
print("=" * 80)

# Check for duplicates
duplicates = df_clean.duplicated().sum()
print(f"\n🔍 Duplicate Records: {duplicates}")
if duplicates > 0:
    df_clean = df_clean.drop_duplicates()
    print(f"   ✓ Removed {duplicates} duplicate records")

# Create binary tsunami indicator (for easier analysis)
df_clean['Tsunami_Binary'] = (df_clean['Tsunami'] == 'Yes').astype(int)

# Create magnitude categories
def categorize_magnitude(mag):
    if mag < 5.0:
        return 'Minor (< 5.0)'
    elif mag < 6.0:
        return 'Moderate (5.0-5.9)'
    elif mag < 7.0:
        return 'Strong (6.0-6.9)'
    elif mag < 8.0:
        return 'Major (7.0-7.9)'
    else:
        return 'Great (≥ 8.0)'

df_clean['Magnitude_Category'] = df_clean['Magnitude'].apply(categorize_magnitude)

# Create depth categories
def categorize_depth(depth):
    if depth < 70:
        return 'Shallow (< 70 km)'
    elif depth < 300:
        return 'Intermediate (70-300 km)'
    else:
        return 'Deep (> 300 km)'

df_clean['Depth_Category'] = df_clean['Depth_km'].apply(categorize_depth)

print("\n✓ Created derived features:")
print("   - Tsunami_Binary: Numerical indicator of tsunami (0 or 1)")
print("   - Magnitude_Category: Categorical classification of earthquake strength")
print("   - Depth_Category: Categorical classification of earthquake depth")

print("\n✓ Final dataset shape:", df_clean.shape)
print("\nData is ready for analysis! 🎉")

## Section 4: Time-Based Analysis

Explore how earthquake occurrences and tsunami events have changed over the 22-year period (2001–2022).

In [ ]:
# Aggregate data by year
yearly_stats = df_clean.groupby('Year').agg({
    'Magnitude': ['count', 'mean', 'std'],
    'Tsunami_Binary': 'sum',
    'Depth_km': 'mean'
}).round(3)

yearly_stats.columns = ['Earthquake_Count', 'Avg_Magnitude', 'Magnitude_Std', 'Tsunami_Count', 'Avg_Depth']
yearly_stats['Tsunami_Percentage'] = (yearly_stats['Tsunami_Count'] / yearly_stats['Earthquake_Count'] * 100).round(2)

print("=" * 80)
print("YEARLY STATISTICS (2001-2022)")
print("=" * 80)
print(yearly_stats)

# Create visualizations for time-based analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Time-Based Analysis: Earthquake Trends (2001-2022)', fontsize=16, fontweight='bold')

# 1. Earthquake count per year
axes[0, 0].plot(yearly_stats.index, yearly_stats['Earthquake_Count'], 
                marker='o', linewidth=2.5, markersize=8, color='#2E86AB', label='Total Earthquakes')
axes[0, 0].fill_between(yearly_stats.index, yearly_stats['Earthquake_Count'], alpha=0.3, color='#2E86AB')
axes[0, 0].set_title('Annual Earthquake Frequency', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Year', fontsize=11)
axes[0, 0].set_ylabel('Number of Earthquakes', fontsize=11)
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xlim(2000.5, 2022.5)

# 2. Average magnitude per year
axes[0, 1].bar(yearly_stats.index, yearly_stats['Avg_Magnitude'], 
               color='#A23B72', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0, 1].set_title('Average Earthquake Magnitude by Year', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Year', fontsize=11)
axes[0, 1].set_ylabel('Average Magnitude', fontsize=11)
axes[0, 1].grid(True, alpha=0.3, axis='y')
axes[0, 1].set_xlim(2000.5, 2022.5)

# 3. Tsunami count per year
axes[1, 0].bar(yearly_stats.index, yearly_stats['Tsunami_Count'], 
               color='#F18F01', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1, 0].set_title('Annual Tsunami Events', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Year', fontsize=11)
axes[1, 0].set_ylabel('Number of Tsunamis', fontsize=11)
axes[1, 0].grid(True, alpha=0.3, axis='y')
axes[1, 0].set_xlim(2000.5, 2022.5)

# 4. Tsunami percentage per year
axes[1, 1].plot(yearly_stats.index, yearly_stats['Tsunami_Percentage'], 
                marker='s', linewidth=2.5, markersize=7, color='#C73E1D', label='Tsunami %')
axes[1, 1].fill_between(yearly_stats.index, yearly_stats['Tsunami_Percentage'], alpha=0.3, color='#C73E1D')
axes[1, 1].set_title('Percentage of Tsunami-Generating Earthquakes', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Year', fontsize=11)
axes[1, 1].set_ylabel('Tsunami Percentage (%)', fontsize=11)
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_xlim(2000.5, 2022.5)

plt.tight_layout()
plt.show()

# Key insights
print("\n" + "=" * 80)
print("KEY TIME-BASED INSIGHTS")
print("=" * 80)
print(f"\n📅 Analysis Period: {df_clean['Year'].min()} - {df_clean['Year'].max()}")
print(f"📊 Total Earthquakes: {len(df_clean)}")
print(f"🌊 Total Tsunami Events: {(df_clean['Tsunami'] == 'Yes').sum()}")
print(f"📈 Average Earthquakes per Year: {yearly_stats['Earthquake_Count'].mean():.1f}")
print(f"🔝 Peak Year (Earthquakes): {yearly_stats['Earthquake_Count'].idxmax()} ({yearly_stats['Earthquake_Count'].max():.0f} events)")
print(f"🔝 Peak Year (Tsunamis): {yearly_stats['Tsunami_Count'].idxmax()} ({yearly_stats['Tsunami_Count'].max():.0f} events)")
print(f"📊 Average Magnitude: {df_clean['Magnitude'].mean():.2f}")
print(f"🔴 Highest Magnitude Year: {df_clean.groupby('Year')['Magnitude'].max().idxmax()} ({df_clean.groupby('Year')['Magnitude'].max().max():.2f})")

## Section 5: Magnitude and Depth Analysis

Analyze the distribution of earthquake magnitudes and depths, and compare tsunamigenic vs. non-tsunamigenic events.

In [ ]:
# Calculate statistics by tsunami occurrence
tsunami_comparison = df_clean.groupby('Tsunami').agg({
    'Magnitude': ['count', 'mean', 'median', 'std', 'min', 'max'],
    'Depth_km': ['mean', 'median', 'std', 'min', 'max'],
    'Intensity': ['mean', 'std']
}).round(3)

print("=" * 80)
print("MAGNITUDE & DEPTH COMPARISON: TSUNAMI vs. NON-TSUNAMI")
print("=" * 80)
print(tsunami_comparison)

# Identify major earthquakes (≥8.0)
major_eq = df_clean[df_clean['Magnitude'] >= 8.0].sort_values('Magnitude', ascending=False)
print("\n" + "=" * 80)
print(f"MAJOR EARTHQUAKES (Magnitude ≥ 8.0): {len(major_eq)} Events")
print("=" * 80)
if len(major_eq) > 0:
    print(major_eq[['Date', 'Magnitude', 'Depth_km', 'Latitude', 'Longitude', 'Tsunami', 'Tsunami_Magnitude']].to_string())
else:
    print("No earthquakes with magnitude ≥ 8.0 in the dataset")

# Create comprehensive magnitude and depth visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Magnitude and Depth Analysis: Comprehensive Overview', fontsize=16, fontweight='bold')

# 1. Magnitude distribution histogram
axes[0, 0].hist(df_clean['Magnitude'], bins=30, color='#2E86AB', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0, 0].axvline(df_clean['Magnitude'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_clean["Magnitude"].mean():.2f}')
axes[0, 0].axvline(df_clean['Magnitude'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df_clean["Magnitude"].median():.2f}')
axes[0, 0].set_title('Distribution of Earthquake Magnitudes', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Magnitude', fontsize=10)
axes[0, 0].set_ylabel('Frequency', fontsize=10)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Depth distribution histogram
axes[0, 1].hist(df_clean['Depth_km'], bins=30, color='#A23B72', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0, 1].axvline(df_clean['Depth_km'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_clean["Depth_km"].mean():.1f} km')
axes[0, 1].set_title('Distribution of Earthquake Depths', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Depth (km)', fontsize=10)
axes[0, 1].set_ylabel('Frequency', fontsize=10)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Magnitude by Tsunami Status
tsunami_mag = [df_clean[df_clean['Tsunami'] == 'Yes']['Magnitude'].values,
               df_clean[df_clean['Tsunami'] == 'No']['Magnitude'].values]
bp1 = axes[0, 2].boxplot(tsunami_mag, labels=['Tsunami Yes', 'Tsunami No'], patch_artist=True)
for patch, color in zip(bp1['boxes'], ['#F18F01', '#2E86AB']):
    patch.set_facecolor(color)
axes[0, 2].set_title('Magnitude Comparison by Tsunami Status', fontsize=12, fontweight='bold')
axes[0, 2].set_ylabel('Magnitude', fontsize=10)
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Depth by Tsunami Status
tsunami_depth = [df_clean[df_clean['Tsunami'] == 'Yes']['Depth_km'].values,
                 df_clean[df_clean['Tsunami'] == 'No']['Depth_km'].values]
bp2 = axes[1, 0].boxplot(tsunami_depth, labels=['Tsunami Yes', 'Tsunami No'], patch_artist=True)
for patch, color in zip(bp2['boxes'], ['#F18F01', '#2E86AB']):
    patch.set_facecolor(color)
axes[1, 0].set_title('Depth Comparison by Tsunami Status', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Depth (km)', fontsize=10)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 5. Magnitude category distribution
mag_cat_counts = df_clean['Magnitude_Category'].value_counts()
mag_cat_order = ['Minor (< 5.0)', 'Moderate (5.0-5.9)', 'Strong (6.0-6.9)', 'Major (7.0-7.9)', 'Great (≥ 8.0)']
mag_cat_counts = mag_cat_counts.reindex([m for m in mag_cat_order if m in mag_cat_counts.index])
colors_mag = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#8B0000']
axes[1, 1].bar(range(len(mag_cat_counts)), mag_cat_counts.values, color=colors_mag[:len(mag_cat_counts)], 
               edgecolor='black', linewidth=0.5)
axes[1, 1].set_xticks(range(len(mag_cat_counts)))
axes[1, 1].set_xticklabels(mag_cat_counts.index, rotation=45, ha='right', fontsize=9)
axes[1, 1].set_title('Earthquake Magnitude Categories', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Count', fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 6. Depth category distribution
depth_cat_counts = df_clean['Depth_Category'].value_counts()
depth_cat_order = ['Shallow (< 70 km)', 'Intermediate (70-300 km)', 'Deep (> 300 km)']
depth_cat_counts = depth_cat_counts.reindex([d for d in depth_cat_order if d in depth_cat_counts.index])
colors_depth = ['#C73E1D', '#F18F01', '#2E86AB']
axes[1, 2].pie(depth_cat_counts.values, labels=depth_cat_counts.index, autopct='%1.1f%%',
               colors=colors_depth[:len(depth_cat_counts)], startangle=90)
axes[1, 2].set_title('Earthquake Depth Categories Distribution', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("KEY MAGNITUDE & DEPTH INSIGHTS")
print("=" * 80)
tsunami_yes = df_clean[df_clean['Tsunami'] == 'Yes']
tsunami_no = df_clean[df_clean['Tsunami'] == 'No']
print(f"\n🌊 TSUNAMI-GENERATING EARTHQUAKES:")
print(f"   - Average Magnitude: {tsunami_yes['Magnitude'].mean():.2f} (vs. {tsunami_no['Magnitude'].mean():.2f} for non-tsunami)")
print(f"   - Average Depth: {tsunami_yes['Depth_km'].mean():.1f} km (vs. {tsunami_no['Depth_km'].mean():.1f} km for non-tsunami)")
print(f"   - Magnitude Range: {tsunami_yes['Magnitude'].min():.2f} - {tsunami_yes['Magnitude'].max():.2f}")
print(f"   - Depth Range: {tsunami_yes['Depth_km'].min():.1f} - {tsunami_yes['Depth_km'].max():.1f} km")
print(f"\n📊 MAGNITUDE DISTRIBUTION:")
print(df_clean['Magnitude_Category'].value_counts())
print(f"\n📍 DEPTH DISTRIBUTION:")
print(df_clean['Depth_Category'].value_counts())

## Section 6: Geographic Distribution Analysis

Plot earthquake locations on 2D scatter plots to identify geographic patterns and clusters.

In [ ]:
# Identify earthquake clusters by latitude/longitude regions
print("=" * 80)
print("GEOGRAPHIC ANALYSIS: EARTHQUAKE DISTRIBUTION")
print("=" * 80)

# Create regional bins
lat_bins = [-90, -30, 0, 30, 90]
lon_bins = [-180, -120, -60, 0, 60, 120, 180]

df_clean['Lat_Region'] = pd.cut(df_clean['Latitude'], bins=lat_bins, 
                                 labels=['South', 'South-Mid', 'North-Mid', 'North'])
df_clean['Lon_Region'] = pd.cut(df_clean['Longitude'], bins=lon_bins,
                                 labels=['Pacific-W', 'Americas', 'Atlantic', 'Europe-Africa', 'Asia-Pacific', 'Pacific-E'])

# Count by region
regional_dist = df_clean.groupby(['Lat_Region', 'Lon_Region']).size().unstack(fill_value=0)
print("\nEarthquakes by Geographic Region:")
print(regional_dist)

# Create 2D scatter plots
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Geographic Distribution of Earthquakes (2001-2022)', fontsize=16, fontweight='bold')

# Plot 1: All earthquakes with magnitude as size
scatter1 = axes[0].scatter(df_clean['Longitude'], df_clean['Latitude'], 
                          c=df_clean['Magnitude'], s=df_clean['Magnitude']*15, 
                          cmap='YlOrRd', alpha=0.6, edgecolors='black', linewidth=0.3)
axes[0].set_title('Earthquake Locations (Size and Color = Magnitude)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Longitude', fontsize=11)
axes[0].set_ylabel('Latitude', fontsize=11)
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-180, 180)
axes[0].set_ylim(-90, 90)
cbar1 = plt.colorbar(scatter1, ax=axes[0])
cbar1.set_label('Magnitude', fontsize=10)

# Highlight major earthquakes
major_eq_plot = df_clean[df_clean['Magnitude'] >= 7.0]
axes[0].scatter(major_eq_plot['Longitude'], major_eq_plot['Latitude'],
               s=200, marker='*', edgecolors='darkred', linewidth=1.5, 
               facecolors='none', label='Magnitude ≥ 7.0')
axes[0].legend(loc='lower left', fontsize=10)

# Plot 2: Tsunami vs. Non-Tsunami distinction
colors = ['#2E86AB' if t == 'No' else '#F18F01' for t in df_clean['Tsunami']]
scatter2 = axes[1].scatter(df_clean['Longitude'], df_clean['Latitude'],
                          c=colors, s=df_clean['Magnitude']*10, alpha=0.6,
                          edgecolors='black', linewidth=0.3)

# Create custom legend for scatter plot
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2E86AB', edgecolor='black', label='No Tsunami'),
                   Patch(facecolor='#F18F01', edgecolor='black', label='Tsunami Generated')]
axes[1].legend(handles=legend_elements, loc='lower left', fontsize=10)

axes[1].set_title('Tsunami vs. Non-Tsunami Earthquakes', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Longitude', fontsize=11)
axes[1].set_ylabel('Latitude', fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-180, 180)
axes[1].set_ylim(-90, 90)

plt.tight_layout()
plt.show()

# Regional tsunami analysis
print("\n" + "=" * 80)
print("REGIONAL TSUNAMI ANALYSIS")
print("=" * 80)

region_tsunami = df_clean.groupby('Lon_Region')['Tsunami_Binary'].agg(['sum', 'count'])
region_tsunami['Percentage'] = (region_tsunami['sum'] / region_tsunami['count'] * 100).round(2)
region_tsunami.columns = ['Tsunami_Count', 'Total_Count', 'Tsunami_Percentage']
print("\nTsunami Distribution by Longitude Region:")
print(region_tsunami.sort_values('Tsunami_Percentage', ascending=False))

# Identify high-risk zones (high tsunami frequency)
high_risk_zones = df_clean[df_clean['Tsunami'] == 'Yes'].groupby('Lon_Region').size().sort_values(ascending=False)
print("\n🌊 High-Risk Zones (by Tsunami Count):")
print(high_risk_zones.head(5))

# Find clusters of tsunami events
print("\n" + "=" * 80)
print("TSUNAMI EVENT CLUSTERS")
print("=" * 80)
tsunami_events = df_clean[df_clean['Tsunami'] == 'Yes'].sort_values('Magnitude', ascending=False)
print(tsunami_events[['Date', 'Latitude', 'Longitude', 'Magnitude', 'Tsunami_Magnitude', 'Death_Toll']].head(10).to_string())

## Section 7: Statistical and Comparative Analysis

Create detailed box plots and bar charts to compare seismic features between tsunami and non-tsunami events.

In [ ]:
# Comprehensive statistical analysis
print("=" * 80)
print("STATISTICAL COMPARISON: TSUNAMI vs. NON-TSUNAMI")
print("=" * 80)

from scipy import stats

# Perform statistical tests
tsunami_yes = df_clean[df_clean['Tsunami'] == 'Yes']
tsunami_no = df_clean[df_clean['Tsunami'] == 'No']

# T-tests for magnitude and depth
t_stat_mag, p_val_mag = stats.ttest_ind(tsunami_yes['Magnitude'], tsunami_no['Magnitude'])
t_stat_depth, p_val_depth = stats.ttest_ind(tsunami_yes['Depth_km'], tsunami_no['Depth_km'])

print(f"\nMagnitude T-Test:")
print(f"  t-statistic: {t_stat_mag:.4f}, p-value: {p_val_mag:.4e}")
print(f"  Interpretation: {'Significant difference' if p_val_mag < 0.05 else 'No significant difference'} in magnitude between groups")

print(f"\nDepth T-Test:")
print(f"  t-statistic: {t_stat_depth:.4f}, p-value: {p_val_depth:.4e}")
print(f"  Interpretation: {'Significant difference' if p_val_depth < 0.05 else 'No significant difference'} in depth between groups")

# Mann-Whitney U test (non-parametric alternative)
u_stat_mag, p_val_u_mag = stats.mannwhitneyu(tsunami_yes['Magnitude'], tsunami_no['Magnitude'])
u_stat_depth, p_val_u_depth = stats.mannwhitneyu(tsunami_yes['Depth_km'], tsunami_no['Depth_km'])

print(f"\nMann-Whitney U Tests (Non-parametric):")
print(f"  Magnitude U: {u_stat_mag:.1f}, p-value: {p_val_u_mag:.4e}")
print(f"  Depth U: {u_stat_depth:.1f}, p-value: {p_val_u_depth:.4e}")

# Create comprehensive comparative visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Statistical Comparison: Tsunami vs. Non-Tsunami Earthquakes', fontsize=16, fontweight='bold')

# 1. Magnitude comparison (violin plot)
parts1 = axes[0, 0].violinplot([tsunami_no['Magnitude'].values, tsunami_yes['Magnitude'].values],
                               positions=[1, 2], showmeans=True, showmedians=True)
axes[0, 0].set_xticks([1, 2])
axes[0, 0].set_xticklabels(['No Tsunami', 'Tsunami'])
axes[0, 0].set_title('Magnitude Distribution (Violin Plot)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Magnitude', fontsize=10)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# 2. Depth comparison (violin plot)
parts2 = axes[0, 1].violinplot([tsunami_no['Depth_km'].values, tsunami_yes['Depth_km'].values],
                               positions=[1, 2], showmeans=True, showmedians=True)
axes[0, 1].set_xticks([1, 2])
axes[0, 1].set_xticklabels(['No Tsunami', 'Tsunami'])
axes[0, 1].set_title('Depth Distribution (Violin Plot)', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Depth (km)', fontsize=10)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# 3. Intensity comparison
axes[0, 2].hist([tsunami_no['Intensity'].values, tsunami_yes['Intensity'].values],
               label=['No Tsunami', 'Tsunami'], bins=12, color=['#2E86AB', '#F18F01'], alpha=0.7, edgecolor='black')
axes[0, 2].set_title('Earthquake Intensity Distribution', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Intensity', fontsize=10)
axes[0, 2].set_ylabel('Frequency', fontsize=10)
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3, axis='y')

# 4. Magnitude categories with tsunami percentage
mag_cat_tsunami = pd.crosstab(df_clean['Magnitude_Category'], df_clean['Tsunami'])
mag_cat_order = ['Minor (< 5.0)', 'Moderate (5.0-5.9)', 'Strong (6.0-6.9)', 'Major (7.0-7.9)', 'Great (≥ 8.0)']
mag_cat_tsunami = mag_cat_tsunami.reindex([m for m in mag_cat_order if m in mag_cat_tsunami.index])

x_pos = np.arange(len(mag_cat_tsunami))
width = 0.35
axes[1, 0].bar(x_pos - width/2, mag_cat_tsunami['No'], width, label='No Tsunami', 
              color='#2E86AB', edgecolor='black', linewidth=0.5)
axes[1, 0].bar(x_pos + width/2, mag_cat_tsunami['Yes'], width, label='Tsunami', 
              color='#F18F01', edgecolor='black', linewidth=0.5)
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(mag_cat_tsunami.index, rotation=45, ha='right', fontsize=9)
axes[1, 0].set_title('Tsunami Occurrence by Magnitude Category', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Count', fontsize=10)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 5. Depth categories with tsunami percentage
depth_cat_tsunami = pd.crosstab(df_clean['Depth_Category'], df_clean['Tsunami'])
depth_cat_order = ['Shallow (< 70 km)', 'Intermediate (70-300 km)', 'Deep (> 300 km)']
depth_cat_tsunami = depth_cat_tsunami.reindex([d for d in depth_cat_order if d in depth_cat_tsunami.index])

x_pos2 = np.arange(len(depth_cat_tsunami))
axes[1, 1].bar(x_pos2 - width/2, depth_cat_tsunami['No'], width, label='No Tsunami',
              color='#2E86AB', edgecolor='black', linewidth=0.5)
axes[1, 1].bar(x_pos2 + width/2, depth_cat_tsunami['Yes'], width, label='Tsunami',
              color='#F18F01', edgecolor='black', linewidth=0.5)
axes[1, 1].set_xticks(x_pos2)
axes[1, 1].set_xticklabels(depth_cat_tsunami.index, rotation=45, ha='right', fontsize=9)
axes[1, 1].set_title('Tsunami Occurrence by Depth Category', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Count', fontsize=10)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

# 6. Tsunami percentage by depth category
depth_cat_tsunami_pct = (depth_cat_tsunami['Yes'] / (depth_cat_tsunami['Yes'] + depth_cat_tsunami['No']) * 100).sort_values(ascending=False)
colors_pct = ['#C73E1D' if pct > 30 else '#F18F01' if pct > 10 else '#2E86AB' for pct in depth_cat_tsunami_pct.values]
axes[1, 2].barh(range(len(depth_cat_tsunami_pct)), depth_cat_tsunami_pct.values, color=colors_pct, edgecolor='black', linewidth=0.5)
axes[1, 2].set_yticks(range(len(depth_cat_tsunami_pct)))
axes[1, 2].set_yticklabels(depth_cat_tsunami_pct.index, fontsize=10)
axes[1, 2].set_title('Tsunami Percentage by Depth Category', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Percentage (%)', fontsize=10)
axes[1, 2].grid(True, alpha=0.3, axis='x')

for i, v in enumerate(depth_cat_tsunami_pct.values):
    axes[1, 2].text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("SUMMARY STATISTICS BY TSUNAMI STATUS")
print("=" * 80)
summary_stats = df_clean.groupby('Tsunami')[['Magnitude', 'Depth_km', 'Intensity', 'Death_Toll', 'Economic_Impact_Million_USD']].describe().round(2)
print(summary_stats)

## Section 8: Correlation Analysis

Analyze correlations between seismic variables using correlation matrices and heatmaps.

In [ ]:
# Prepare data for correlation analysis
correlation_columns = ['Magnitude', 'Depth_km', 'Latitude', 'Longitude', 
                       'Intensity', 'Tsunami_Binary', 'Tsunami_Magnitude', 
                       'Wave_Height_m', 'Death_Toll', 'Economic_Impact_Million_USD']

df_corr = df_clean[correlation_columns].copy()

# Calculate correlation matrix
correlation_matrix = df_corr.corr()

print("=" * 80)
print("CORRELATION ANALYSIS")
print("=" * 80)
print("\nFull Correlation Matrix:")
print(correlation_matrix.round(3))

# Extract correlations with Tsunami_Binary
print("\n" + "=" * 80)
print("CORRELATIONS WITH TSUNAMI OCCURRENCE")
print("=" * 80)
tsunami_correlations = correlation_matrix['Tsunami_Binary'].sort_values(ascending=False)
print(tsunami_correlations)

# Identify strong correlations (Pearson r > 0.5 or < -0.5)
print("\n" + "=" * 80)
print("STRONG CORRELATIONS (|r| > 0.5)")
print("=" * 80)
strong_corr = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        if abs(correlation_matrix.iloc[i, j]) > 0.5:
            strong_corr.append({
                'Variable 1': correlation_matrix.columns[i],
                'Variable 2': correlation_matrix.columns[j],
                'Correlation': correlation_matrix.iloc[i, j]
            })

if strong_corr:
    strong_corr_df = pd.DataFrame(strong_corr).sort_values('Correlation', 
                                                            key=abs, ascending=False)
    print(strong_corr_df.to_string())
else:
    print("No strong correlations found (|r| > 0.5)")

# Create heatmap visualizations
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Correlation Analysis of Seismic Variables', fontsize=16, fontweight='bold')

# Heatmap 1: Full correlation matrix
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=axes[0],
            vmin=-1, vmax=1, annot_kws={'fontsize': 8})
axes[0].set_title('Complete Correlation Matrix', fontsize=13, fontweight='bold')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right', fontsize=9)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0, fontsize=9)

# Heatmap 2: Focused on key variables with Tsunami
key_variables = ['Magnitude', 'Depth_km', 'Latitude', 'Longitude', 
                 'Intensity', 'Tsunami_Binary', 'Tsunami_Magnitude', 'Wave_Height_m']
focused_corr = df_clean[key_variables].corr()

sns.heatmap(focused_corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8}, ax=axes[1],
            vmin=-1, vmax=1, annot_kws={'fontsize': 9})
axes[1].set_title('Focused Correlation (Key Variables)', fontsize=13, fontweight='bold')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right', fontsize=9)
axes[1].set_yticklabels(axes[1].get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.show()

# Scatter plots showing key relationships
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Key Relationships in the Data', fontsize=16, fontweight='bold')

# Scatter 1: Magnitude vs. Depth colored by Tsunami
for tsunami_val, color in [('No', '#2E86AB'), ('Yes', '#F18F01')]:
    mask = df_clean['Tsunami'] == tsunami_val
    axes[0, 0].scatter(df_clean[mask]['Magnitude'], df_clean[mask]['Depth_km'],
                      alpha=0.5, s=50, color=color, label=f'Tsunami: {tsunami_val}', edgecolors='black', linewidth=0.3)
axes[0, 0].set_title('Magnitude vs. Depth', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Magnitude', fontsize=10)
axes[0, 0].set_ylabel('Depth (km)', fontsize=10)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Scatter 2: Magnitude vs. Wave Height
tsunami_yes = df_clean[df_clean['Tsunami'] == 'Yes']
axes[0, 1].scatter(tsunami_yes['Magnitude'], tsunami_yes['Wave_Height_m'],
                   alpha=0.6, s=80, color='#F18F01', edgecolors='black', linewidth=0.5)
axes[0, 1].set_title('Magnitude vs. Wave Height (Tsunamis Only)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Magnitude', fontsize=10)
axes[0, 1].set_ylabel('Wave Height (m)', fontsize=10)
axes[0, 1].grid(True, alpha=0.3)

# Scatter 3: Depth vs. Wave Height
axes[0, 2].scatter(tsunami_yes['Depth_km'], tsunami_yes['Wave_Height_m'],
                   alpha=0.6, s=80, color='#A23B72', edgecolors='black', linewidth=0.5)
axes[0, 2].set_title('Depth vs. Wave Height (Tsunamis Only)', fontsize=12, fontweight='bold')
axes[0, 2].set_xlabel('Depth (km)', fontsize=10)
axes[0, 2].set_ylabel('Wave Height (m)', fontsize=10)
axes[0, 2].grid(True, alpha=0.3)

# Scatter 4: Magnitude vs. Death Toll
axes[1, 0].scatter(df_clean['Magnitude'], df_clean['Death_Toll'],
                   alpha=0.5, s=60, color='#C73E1D', edgecolors='black', linewidth=0.3)
axes[1, 0].set_title('Magnitude vs. Death Toll', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Magnitude', fontsize=10)
axes[1, 0].set_ylabel('Death Toll', fontsize=10)
axes[1, 0].grid(True, alpha=0.3)

# Scatter 5: Magnitude vs. Economic Impact
axes[1, 1].scatter(df_clean['Magnitude'], df_clean['Economic_Impact_Million_USD'],
                   alpha=0.5, s=60, color='#2E86AB', edgecolors='black', linewidth=0.3)
axes[1, 1].set_title('Magnitude vs. Economic Impact', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Magnitude', fontsize=10)
axes[1, 1].set_ylabel('Economic Impact (Million USD)', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

# Scatter 6: Depth vs. Intensity
axes[1, 2].scatter(df_clean['Depth_km'], df_clean['Intensity'],
                   alpha=0.5, s=60, color='#F18F01', edgecolors='black', linewidth=0.3)
axes[1, 2].set_title('Depth vs. Intensity', fontsize=12, fontweight='bold')
axes[1, 2].set_xlabel('Depth (km)', fontsize=10)
axes[1, 2].set_ylabel('Intensity', fontsize=10)
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Section 9: Key Findings and Insights

Comprehensive summary of the exploratory data analysis findings.

In [ ]:
print("=" * 100)
print("KEY FINDINGS AND INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("=" * 100)

# Key Statistics
total_earthquakes = len(df_clean)
total_tsunamis = (df_clean['Tsunami'] == 'Yes').sum()
tsunami_rate = total_tsunamis / total_earthquakes * 100
avg_mag = df_clean['Magnitude'].mean()
avg_depth = df_clean['Depth_km'].mean()

print(f"\n📊 DATASET OVERVIEW:")
print(f"   • Total Earthquakes Analyzed: {total_earthquakes:,}")
print(f"   • Time Period: {df_clean['Year'].min()}-{df_clean['Year'].max()} (22 years)")
print(f"   • Tsunami-Generating Events: {total_tsunamis} ({tsunami_rate:.2f}%)")
print(f"   • Average Magnitude: {avg_mag:.2f}")
print(f"   • Average Depth: {avg_depth:.1f} km")

# Magnitude insights
print(f"\n🔴 MAGNITUDE ANALYSIS:")
mag_stats = df_clean.groupby('Magnitude_Category')['Magnitude_Category'].count()
print(f"   • Minor earthquakes (< 5.0): {mag_stats.get('Minor (< 5.0)', 0)} ({mag_stats.get('Minor (< 5.0)', 0)/len(df_clean)*100:.1f}%)")
print(f"   • Moderate (5.0-5.9): {mag_stats.get('Moderate (5.0-5.9)', 0)} ({mag_stats.get('Moderate (5.0-5.9)', 0)/len(df_clean)*100:.1f}%)")
print(f"   • Strong (6.0-6.9): {mag_stats.get('Strong (6.0-6.9)', 0)} ({mag_stats.get('Strong (6.0-6.9)', 0)/len(df_clean)*100:.1f}%)")
print(f"   • Major (7.0-7.9): {mag_stats.get('Major (7.0-7.9)', 0)} ({mag_stats.get('Major (7.0-7.9)', 0)/len(df_clean)*100:.1f}%)")
print(f"   • Great (≥ 8.0): {mag_stats.get('Great (≥ 8.0)', 0)} ({mag_stats.get('Great (≥ 8.0)', 0)/len(df_clean)*100:.1f}%)")

# Depth insights
print(f"\n📍 DEPTH ANALYSIS:")
depth_stats = df_clean.groupby('Depth_Category')['Depth_Category'].count()
print(f"   • Shallow (< 70 km): {depth_stats.get('Shallow (< 70 km)', 0)} ({depth_stats.get('Shallow (< 70 km)', 0)/len(df_clean)*100:.1f}%)")
print(f"   • Intermediate (70-300 km): {depth_stats.get('Intermediate (70-300 km)', 0)} ({depth_stats.get('Intermediate (70-300 km)', 0)/len(df_clean)*100:.1f}%)")
print(f"   • Deep (> 300 km): {depth_stats.get('Deep (> 300 km)', 0)} ({depth_stats.get('Deep (> 300 km)', 0)/len(df_clean)*100:.1f}%)")

# Tsunami characteristics
tsunami_yes = df_clean[df_clean['Tsunami'] == 'Yes']
tsunami_no = df_clean[df_clean['Tsunami'] == 'No']

print(f"\n🌊 TSUNAMI-GENERATING vs. NON-TSUNAMI EARTHQUAKES:")
print(f"\n   MAGNITUDE COMPARISON:")
print(f"   • Tsunami-generating avg magnitude: {tsunami_yes['Magnitude'].mean():.2f} ± {tsunami_yes['Magnitude'].std():.2f}")
print(f"   • Non-tsunami avg magnitude: {tsunami_no['Magnitude'].mean():.2f} ± {tsunami_no['Magnitude'].std():.2f}")
print(f"   • Difference: {tsunami_yes['Magnitude'].mean() - tsunami_no['Magnitude'].mean():.2f} magnitude points")

print(f"\n   DEPTH COMPARISON:")
print(f"   • Tsunami-generating avg depth: {tsunami_yes['Depth_km'].mean():.1f} ± {tsunami_yes['Depth_km'].std():.1f} km")
print(f"   • Non-tsunami avg depth: {tsunami_no['Depth_km'].mean():.1f} ± {tsunami_no['Depth_km'].std():.1f} km")
print(f"   • Difference: {tsunami_yes['Depth_km'].mean() - tsunami_no['Depth_km'].mean():.1f} km")

# Threshold analysis
print(f"\n⚠️  TSUNAMI RISK THRESHOLDS:")
for mag_threshold in [6.0, 6.5, 7.0, 7.5, 8.0]:
    eq_above = len(df_clean[df_clean['Magnitude'] >= mag_threshold])
    tsunami_above = len(df_clean[(df_clean['Magnitude'] >= mag_threshold) & (df_clean['Tsunami'] == 'Yes')])
    if eq_above > 0:
        pct = tsunami_above / eq_above * 100
        print(f"   • Magnitude ≥ {mag_threshold}: {pct:.1f}% tsunami rate ({tsunami_above}/{eq_above} events)")

for depth_threshold in [30, 50, 70, 100, 150]:
    eq_below = len(df_clean[df_clean['Depth_km'] <= depth_threshold])
    tsunami_below = len(df_clean[(df_clean['Depth_km'] <= depth_threshold) & (df_clean['Tsunami'] == 'Yes')])
    if eq_below > 0:
        pct = tsunami_below / eq_below * 100
        print(f"   • Depth ≤ {depth_threshold} km: {pct:.1f}% tsunami rate ({tsunami_below}/{eq_below} events)")

# Impact analysis
print(f"\n💔 IMPACT ASSESSMENT:")
total_deaths = df_clean['Death_Toll'].sum()
total_econ_loss = df_clean['Economic_Impact_Million_USD'].sum()
max_deaths_eq = df_clean.loc[df_clean['Death_Toll'].idxmax()]
max_econ_eq = df_clean.loc[df_clean['Economic_Impact_Million_USD'].idxmax()]

print(f"   • Total Death Toll: {total_deaths:,}")
print(f"   • Total Economic Loss: ${total_econ_loss:,.0f} Million")
print(f"   • Deadliest Event: {max_deaths_eq['Date'].strftime('%Y-%m-%d')} "
      f"(Magnitude {max_deaths_eq['Magnitude']:.2f}, Deaths: {int(max_deaths_eq['Death_Toll'])})")
print(f"   • Highest Economic Loss: {max_econ_eq['Date'].strftime('%Y-%m-%d')} "
      f"(Magnitude {max_econ_eq['Magnitude']:.2f}, Loss: ${max_econ_eq['Economic_Impact_Million_USD']:.0f}M)")

# Regional insights
print(f"\n🌍 GEOGRAPHIC PATTERNS:")
high_activity_regions = df_clean.groupby('Lon_Region').size().sort_values(ascending=False).head(3)
high_tsunami_regions = df_clean[df_clean['Tsunami'] == 'Yes'].groupby('Lon_Region').size().sort_values(ascending=False).head(3)

print(f"   Top 3 High-Activity Regions:")
for i, (region, count) in enumerate(high_activity_regions.items(), 1):
    pct = count / len(df_clean) * 100
    print(f"   {i}. {region}: {count} events ({pct:.1f}%)")

print(f"\n   Top 3 High-Tsunami Regions:")
for i, (region, count) in enumerate(high_tsunami_regions.items(), 1):
    total = len(df_clean[df_clean['Lon_Region'] == region])
    pct = count / total * 100 if total > 0 else 0
    print(f"   {i}. {region}: {count} tsunamis ({pct:.1f}% of {total} events)")

# Correlation insights
print(f"\n🔗 CORRELATION INSIGHTS:")
tsunami_magnitude_corr = df_clean['Tsunami_Binary'].corr(df_clean['Magnitude'])
tsunami_depth_corr = df_clean['Tsunami_Binary'].corr(df_clean['Depth_km'])
depth_magnitude_corr = df_clean['Magnitude'].corr(df_clean['Depth_km'])

print(f"   • Tsunami vs. Magnitude: r = {tsunami_magnitude_corr:.3f} (Strong positive correlation)")
print(f"   • Tsunami vs. Depth: r = {tsunami_depth_corr:.3f} (Strong negative correlation)")
print(f"   • Magnitude vs. Depth: r = {depth_magnitude_corr:.3f}")
print(f"   ➜ Interpretation: Larger magnitude & shallower earthquakes → Higher tsunami probability")

# Summary statistics
print(f"\n📈 TIME-BASED TRENDS:")
yearly_counts = df_clean.groupby('Year').size()
uptrend_years = (yearly_counts.iloc[-5:].mean() > yearly_counts.iloc[:5].mean())
print(f"   • Trend (Last 5 vs. First 5 years): {'↑ Increasing' if uptrend_years else '↓ Decreasing'}")
print(f"   • Average events per year: {yearly_counts.mean():.1f}")
print(f"   • Peak activity year: {yearly_counts.idxmax()} ({yearly_counts.max()} events)")
print(f"   • Lowest activity year: {yearly_counts.idxmin()} ({yearly_counts.min()} events)")

print("\n" + "=" * 100)
print("END OF ANALYSIS")
print("=" * 100)